The notebook mirros the train_from_scratch.py which is main about llama3 structure + MLA and focus on DSA part integration.

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import math
import random
from typing import List, Optional, Tuple, Union, Literal
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
from torch import nn
import os
from torch.utils.data import IterableDataset, Dataset
import json
import numpy as np
from transformers import PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast
from transformers import PretrainedConfig
from transformers import (Trainer, TrainingArguments, AutoModelForCausalLM, AutoTokenizer,
                          DefaultDataCollator, DataCollatorForTokenClassification, AutoConfig,
                          TextStreamer)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

/Users/yingyao/miniconda3/envs/transformer-practice/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Pre-Training

In [4]:
# RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        # rsqrt = 1/sqrt(x)
        result = self.weight * (hidden_states * torch.rsqrt(torch.mean(hidden_states * hidden_states, dim=-1, keepdim=True) + self.variance_epsilon))
        return result

In [5]:
# RoPE
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotate_pos_emb(q, k, cos, sin, unsqueeze_dim=2):
    
    cos = cos.unsqueeze(unsqueeze_dim) # (1, seq_len, 1, dim)
    sin = sin.unsqueeze(unsqueeze_dim) # (1, seq_len, 1, dim)
   
    q_embed = (q*cos) + (rotate_half(q)*sin)  # (batch_size, seq_len, head_num, dim) * (1, seq_len, 1, dim) = (batch_size, seq_len, head_num, dim) 广播
    k_embed = (k*cos) + (rotate_half(k)*sin)  # (batch_size, seq_len, head_num, dim) * (1, seq_len, 1, dim) = = (batch_size, seq_len, head_num, dim) 广播
    
    return q_embed, k_embed

class RotaryEmbedding(nn.Module):
    # Here is slight different than the slides, originally is [x1, x2, x3, x4,. ...] * [cos(m * theta_1), cos(m * theta_1), cos(m * theta_2), cos(m * theta_2), ..cos(m * theta_d/2)] 
    # + [-x2, x1, -x4, x3, ...] * [sin(m * theta_1), sin(m * theta_1), sin(m * theta_2), sin(m * theta_2), ..sin(m * theta_d/2)], which is equivalently to be 
    # [x1, x3, ..., x2, x4,. ...] * [cos(m * theta_1), cos(m * theta_2), ..., cos(m * theta_1), cos(m * theta_2), ..cos(m * theta_d/2)] 
    # + [-x2, -x4,..., x1, x3, ...] * [sin(m * theta_1), sin(m * theta_2), ..., sin(m * theta_1), sin(m * theta_2), ..sin(m * theta_d/2)]
    def __init__(self, dim, max_seq_len=2048):
        super(RotaryEmbedding, self).__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))  # (dim/2)
        t = torch.arange(max_seq_len).float().unsqueeze(1)  # (max_seq_len, 1)
        freqs = t @ inv_freq.unsqueeze(0)  #(max_seq_len, 1)*(1, dim/2) = (max_seq_len, dim/2), e.g. m * theta_i part in the slides
        freqs = torch.cat((freqs, freqs), dim=-1)  # (max_seq_len, dim)
        
        self.register_buffer("cos_cached", freqs.cos())
        self.register_buffer("sin_cached", freqs.sin())
        
    def forward(self, q, k, start_pos=0):
        # During decode with KV cache, the new token sits at absolute position `start_pos`,
        # so slice cos/sin from there, not from 0. With start_pos=0 this matches old behavior.
        seq_len = q.shape[1]
        cos = self.cos_cached[start_pos:start_pos+seq_len, :].unsqueeze(0)  # (1, seq_len, dim)
        sin = self.sin_cached[start_pos:start_pos+seq_len, :].unsqueeze(0)  # (1, seq_len, dim)
        return apply_rotate_pos_emb(q, k, cos, sin)
    

In [6]:
world_size = 1
rank = 0
block_size = 128
gemm_impl: Literal["bf16", "fp8"] = "bf16"
attn_impl: Literal["naive", "absorb"] = "absorb"

In [ ]:
# Config
def default_attn_schedule(n_layers) -> List[Literal["hca", "csa", "mla"]]:
    # V4-Pro-style, suppp
    sched = []
    for i in range(n_layers):
        if i < 2:            sched.append("hca")
        elif i % 2 == 0:     sched.append("csa")
        else:                sched.append("mla")
    return sched


class Config(PretrainedConfig):
    model_type = "v4_replica" 

    def __init__(
        self,
        vocab_size=6400,
        hidden_size=512,
        n_layers = 8,
        num_attention_heads=16,
        num_key_value_heads = 8,
        flash_attn = True,
        attention_bias = False,
        max_batch_size: int = 2,
        max_seq_len = 2048,
        intermediate_size = 2048,
        mlp_bias = False,
        dropout = 0.0,

        dtype: Literal["bf16", "fp8"] = "bf16",
        moe_intermediate_size = 256,
        
        # moe
        n_routed_experts: int = 64,
        n_shared_experts: int = 2,
        n_activated_experts: int = 6,
        n_expert_groups: int = 1,
        n_limited_groups: int = 1,
        score_func: Literal["softmax", "sigmoid"] = "softmax",
        route_scale: float = 1.,
        
        # mla
        q_lora_rank: int = 0,
        kv_lora_rank: int = 128,
        qk_nope_head_dim: int = 32,   # hidden_size/num_attention_heads, 512/16
        qk_rope_head_dim: int = 16,  # the dim size of rotary embedding per head
        v_head_dim: int = 32,  # usually same as qk_nope_head_dim
        
        # dsa - lightening indexer
        index_n_heads: int = 4,
        index_head_dim: int = 32,
        index_rope_head_dim: int = 16,
        index_topk: int = 32,
        share_q_lora_with_indexer: bool = True,  # feed indexer from main MLA's q-lora projection

        # dsa - training stage params
        dsa_stage: Literal["off", "dense_warmup", "sparse"] = "off",
        indexer_kl_weight: float = 1.0,   # multiplier on KL loss when added to LM loss

        # csa 
        compress_rate: int = 4,
        index_block_topk: int = 4,
        compress_head_dim: int = 64,
        compress_n_heads: int = 8,
        # out_proj_group_dim = hidden_size // out_proj_groups
        compress_rope_dim: int = 16,
        sliding_window: int = 64,
        use_attention_sink: bool = True,
        hca_compress_rate: int = 32,
        attn_schedule: list[str] | None = None,

        o_groups: int = 2,
        o_lora_rank: int = 64,
    
        # # yarn
        # original_seq_len: int = 4096,
        # rope_theta: float = 10000.0,
        # rope_factor: float = 40,
        # beta_fast: int = 32,
        # beta_slow: int = 1,
        # mscale: float = 1.,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads
        self.flash_attn = flash_attn
        self.attention_bias = attention_bias
        self.max_batch_size = max_batch_size
        self.max_seq_len = max_seq_len
        self.intermediate_size = intermediate_size
        self.mlp_bias = mlp_bias
        self.dropout = dropout
        self.dtype = dtype
        self.moe_intermediate_size = moe_intermediate_size
        self.n_routed_experts = n_routed_experts
        self.n_shared_experts = n_shared_experts
        self.n_activated_experts = n_activated_experts
        self.n_expert_groups = n_expert_groups 
        self.n_limited_groups = n_limited_groups
        self.score_func = score_func
        self.route_scale = route_scale
        self.q_lora_rank = q_lora_rank
        self.kv_lora_rank = kv_lora_rank
        self.qk_nope_head_dim = qk_nope_head_dim
        self.qk_rope_head_dim = qk_rope_head_dim
        self.v_head_dim = v_head_dim
        self.index_n_heads = index_n_heads
        self.index_head_dim = index_head_dim
        self.index_rope_head_dim = index_rope_head_dim
        self.index_topk = index_topk
        self.share_q_lora_with_indexer = share_q_lora_with_indexer
        self.dsa_stage = dsa_stage
        self.indexer_kl_weight = indexer_kl_weight
        self.compress_rate = compress_rate
        self.index_block_topk = index_block_topk
        self.compress_head_dim = compress_head_dim 
        self.compress_n_heads = compress_n_heads
        self.compress_rope_dim = compress_rope_dim  
        self.sliding_window = sliding_window
        self.use_attention_sink = use_attention_sink
        self.hca_compress_rate = hca_compress_rate
        # auto-fill the per-layer attention schedule when not explicitly provided
        self.attn_schedule = attn_schedule if attn_schedule is not None else default_attn_schedule(n_layers)
        self.o_groups = o_groups
        self.o_lora_rank = o_lora_rank
        
        # self.original_seq_len = original_seq_len
        # self.rope_theta = rope_theta
        # self.rope_factor = rope_factor
        # self.beta_fast = beta_fast
        # self.beta_slow = beta_slow
        # self.mscale = mscale

        assert self.hca_compress_rate % self.compress_rate == 0, \
            "hca_compress_rate must be a multiple of compress_rate (lcm cache-block alignment, Fig 6)"
        assert self.compress_rope_dim <= self.compress_head_dim and self.compress_rope_dim % 2 == 0, \
            "compress_rope_dim must be even and <= compress_head_dim"
        assert self.index_block_topk < self.max_seq_len // self.compress_rate, \
            "index_block_topk must be < number of compressed blocks (max_seq_len // compress_rate)"
        assert len(self.attn_schedule) == self.n_layers, \
            "attn_schedule length must match n_layers"
config = Config()
config.dropout

0.0

In [8]:
# Token compressor (DeepSeek-V4 §2.3.1) — fused single-path: prefill and decode are
# the SAME streaming op, driven by use_kv_cache + start_pos (decode = chunk of S=1).
class TokenCompressor(nn.Module):
    """Compress every `m` tokens into ONE KV entry (DeepSeek-V4 §2.3.1).

    Projections (a = own-block stream, b = previous-block stream, overlap only):
        w_kv / w_z / bias       - value, logit, per-position-in-block logit bias
        w_kv_b / w_z_b / bias_b

    State buffers (only touched when use_kv_cache=True; never checkpointed):
        comp_cache      - finished block entries (the actual KV cache)
        a_val / a_logit - the carried partial block, slots [0, start_pos % m)
        b_val / b_logit - same, for the overlap stream (overlap only)
        prev_b_*        - the previous complete block's b-stream (overlap only)
    """

    def __init__(self, config: Config, compression_rate: int, overlapping: bool, head_dim: int = None):
        super().__init__()
        self.m = compression_rate
        self.c = head_dim if head_dim is not None else config.compress_head_dim  # main or indexer's
        self.overlapping = overlapping
        self.d = d = config.hidden_size

        self.w_kv = nn.Linear(d, self.c, bias=False)            # own-block value (C^a)
        self.w_z  = nn.Linear(d, self.c, bias=False)            # own-block logit (Z^a)
        self.bias = nn.Parameter(torch.zeros(self.m, self.c))   # per-position-in-block bias (B^a)
        if overlapping:
            self.w_kv_b = nn.Linear(d, self.c, bias=False)      # previous-block value (C^b)
            self.w_z_b  = nn.Linear(d, self.c, bias=False)      # previous-block logit (Z^b)
            self.bias_b = nn.Parameter(torch.zeros(self.m, self.c))

        mb, nblk = config.max_batch_size, config.max_seq_len // self.m
        zeros = lambda *s: torch.zeros(*s)
        self.register_buffer("comp_cache", zeros(mb, nblk, self.c),   persistent=False)
        self.register_buffer("c_a",      zeros(mb, self.m, self.c), persistent=False)
        self.register_buffer("z_a",    zeros(mb, self.m, self.c), persistent=False)
        if overlapping:
            ninf = lambda *s: torch.full(s, float('-inf'))
            self.register_buffer("c_b",        zeros(mb, self.m, self.c), persistent=False)
            self.register_buffer("z_b",      ninf(mb, self.m, self.c),  persistent=False)
            self.register_buffer("prev_c_b",   zeros(mb, self.m, self.c), persistent=False)
            self.register_buffer("prev_z_b", ninf(mb, self.m, self.c),  persistent=False)


    def forward(self, hidden_states, use_kv_cache: bool = False, start_pos: int = 0):
        h = hidden_states
        B, S, _ = h.shape
        m, c = self.m, self.c
        carry = use_kv_cache and start_pos > 0          # for decode phase

        slot_ids = (torch.arange(S, device=h.device) + start_pos) % m  # abs_end_pos % m decides to get bias position
        c_a   = self.w_kv(h)                           # (B, S, c)
        z_a = self.w_z(h) + self.bias[slot_ids]
        if self.overlapping:
            c_b   = self.w_kv_b(h)
            z_b = self.w_z_b(h) + self.bias_b[slot_ids]

        # append previous uncompressed info
        filled = start_pos % m if carry else 0
        if filled:
            c_a   = torch.cat([self.c_a[:B, :filled], c_a],   dim=1)
            z_a = torch.cat([self.z_a[:B, :filled], z_a], dim=1)
            if self.overlapping:
                c_b   = torch.cat([self.c_b[:B, :filled], c_b],   dim=1)
                z_b = torch.cat([self.z_b[:B, :filled], z_b], dim=1)

        T  = c_a.size(1)                               # tokens in the contiguo us stream
        nb = T // m                                      # complete blocks to emit now

        # pool each complete block (CSA: append the previous block's b-stream).
        if nb:
            ca = c_a[:, :nb * m].reshape(B, nb, m, c)
            za = z_a[:, :nb * m].reshape(B, nb, m, c)
            if self.overlapping:
                cb = c_b[:, :nb * m].reshape(B, nb, m, c)
                zb = z_b[:, :nb * m].reshape(B, nb, m, c)
                if carry:                                # entry 0's predecessor = carried block
                    prev_cb, prev_zb = self.prev_c_b[:B, None], self.prev_z_b[:B, None] # == unsqueeze(1), i.e. (B, m, c) -> (B, 1, m, c)
                else:                                    # fresh sequence: entry 0 has no predecessor
                    prev_cb = h.new_zeros(B, 1, m, c)
                    prev_zb = h.new_full((B, 1, m, c), float('-inf'))
                ca = torch.cat([ca, torch.cat([prev_cb, cb[:, :-1]], dim=1)], dim=2)  # (B, nb, 2m, c)
                za = torch.cat([za, torch.cat([prev_zb, zb[:, :-1]], dim=1)], dim=2)
            entries = (za.softmax(dim=2) * ca).sum(dim=2)
        else:
            entries = h.new_zeros(B, 0, c)

        if not use_kv_cache:                       
            return entries

        blk0 = start_pos // m
        self.comp_cache[:B, blk0:blk0 + nb] = entries
        if self.overlapping and nb:
            self.prev_c_b[:B]   = cb[:, -1]
            self.prev_z_b[:B] = zb[:, -1]
        keep = T - nb * m
        if keep:
            self.c_a[:B, :keep]   = c_a[:, nb * m:]
            self.z_a[:B, :keep] = z_a[:, nb * m:]
            if self.overlapping:
                self.c_b[:B, :keep]   = c_b[:, nb * m:]
                self.z_b[:B, :keep] = z_b[:, nb * m:]
        return self.comp_cache[:B, :blk0 + nb]


In [9]:
class Indexer(nn.Module):
    """Lightning indexer (DeepSeek-V3.2 DSA / V4 CSA).

    compress_rate=None  -> V3.2 mode: per-token single-head keys (k_t = wk(h_t)); scores token -> token.
    compress_rate=int   -> V4-CSA mode: keys are the OUTPUT of a dedicated TokenCompressor
                           (its own weights, index_head_dim, overlap iff rate==compress_rate),
                           so the indexer scores query-token -> compressed BLOCK (Eq 13-17).
    Returns raw scores I (B, S, T): T = end_pos/S (V3.2) or nb (CSA). The caller applies the causal
    mask + top-k. Prefill only for the CSA key path (decode-phase indexer-key cache = roadmap 6.4).
    """
    def __init__(self, config: Config, compress_rate=None):
        super().__init__()
        self.H_I = config.index_n_heads
        self.d_I = config.index_head_dim
        self.rd  = config.index_rope_head_dim
        self.share_q_lora = config.share_q_lora_with_indexer and config.q_lora_rank > 0
        self.compress_rate = compress_rate

        wq_in = config.q_lora_rank if self.share_q_lora else config.hidden_size
        self.wq = nn.Linear(wq_in, self.H_I * self.d_I)
        self.w_proj = nn.Linear(config.hidden_size, self.H_I)          # per-head gate
        self.rotary_emb = RotaryEmbedding(self.rd, max_seq_len=config.max_seq_len)

        if compress_rate is None:
            # adapt to original DSA
            self.wk = nn.Linear(config.hidden_size, self.d_I)  # single-head per-token key (reduces cache)
            self.register_buffer("index_k_cache",
                torch.zeros(config.max_batch_size, config.max_seq_len, 1, self.d_I), persistent=False) # persistent=False means not loading from a saved checkpoint
        else:
            # adapt to CSA / HCA
            self.token_compressor = TokenCompressor(
                config, compress_rate,
                overlapping=(compress_rate == config.compress_rate), head_dim=self.d_I)
            # no need to further cahce kv in this line as it comes from token compressor

    def _rope_at(self, x, positions):
        # rotate the last rd dims of x (..., T, H, rd) at the given ABSOLUTE positions (T,)
        cos = self.rotary_emb.cos_cached[positions][None, :, None, :]
        sin = self.rotary_emb.sin_cached[positions][None, :, None, :]
        return x * cos + rotate_half(x) * sin

    def forward(self, hidden_states, qr=None, use_kv_cache=False, start_pos=0):
        # `S` query tokens; `end_pos` = prefix length after writing this forward's K. Prefill: S == end_pos.
        B, S, _ = hidden_states.shape
        end_pos = start_pos + S
        dev = hidden_states.device
        q = self.wq(qr if self.share_q_lora else hidden_states).view(B, S, self.H_I, self.d_I)

        if self.compress_rate is None:
            # V3.2: per-token keys, joint q/k RoPE at token positions
            k_new = self.wk(hidden_states).unsqueeze(2)               # (B, S, 1, d_I)
            q_nope, q_rope = q[..., :-self.rd], q[..., -self.rd:]
            k_nope, k_rope = k_new[..., :-self.rd], k_new[..., -self.rd:]
            q_rope, k_rope = self.rotary_emb(q_rope, k_rope, start_pos=start_pos)
            q = torch.cat([q_nope, q_rope], dim=-1)
            k_new = torch.cat([k_nope, k_rope], dim=-1)
            if use_kv_cache:
                self.index_k_cache[:B, start_pos:end_pos] = k_new
                k = self.index_k_cache[:B, :end_pos].squeeze(2)       # (B, end_pos, d_I)
            else:
                k = k_new.squeeze(2)                                  # (B, S, d_I)
        else:
            # V4-CSA
            k = self.token_compressor(hidden_states, use_kv_cache, start_pos).unsqueeze(2)    # (B, nb, 1, d_I)
            nb = k.shape[1]
            q = torch.cat([q[..., :-self.rd],
                           self._rope_at(q[..., -self.rd:], torch.arange(start_pos, end_pos, device=dev))], dim=-1)
            blk_pos = torch.arange(nb, device=dev) * self.compress_rate 
            k = torch.cat([k[..., :-self.rd], self._rope_at(k[..., -self.rd:], blk_pos)], dim=-1)
            k = k.squeeze(2)                                  # (B, S, d_I)

        scores = torch.einsum("bshd,btd->bsht", q, k) / math.sqrt(self.d_I)   # (B, S, H_I, T)
        gated = F.relu(scores) * self.w_proj(hidden_states).unsqueeze(-1)     # (B, S, H_I, T)
        I = gated.sum(dim=2)                                                  # (B, S, T), i.e. T represents as nb after TokenCompressor
        return I

In [10]:
# Indexer smoke test - both modes
torch.manual_seed(0)
Hh = torch.randn(2, 64, config.hidden_size)
# V3.2 per-token mode (keys = wk(h_t)); scores token -> token
idx_tok = Indexer(config)
I_tok = idx_tok(Hh)
assert I_tok.shape == (2, 64, 64), I_tok.shape
# V4-CSA mode: keys = TokenCompressor output (index_head_dim, overlapping); scores token -> block
idx_blk = Indexer(config, compress_rate=config.compress_rate)
I_blk = idx_blk(Hh)
assert I_blk.shape == (2, 64, 64 // config.compress_rate), I_blk.shape   # (2, 64, 16)
assert torch.isfinite(I_tok).all() and torch.isfinite(I_blk).all()
print("Indexer OK - token:", tuple(I_tok.shape), " block:", tuple(I_blk.shape))

Indexer OK - token: (2, 64, 64)  block: (2, 64, 16)


In [11]:
def compute_indexer_kl(attn_probs, indexer_scores, dsa_stage, topk, mask=None):
    p = attn_probs.sum(dim=2)
    p = p / p.sum(dim=-1, keepdim=True).clamp_min(1e-8)
    if dsa_stage == 'dense_warmup':
        log_q = F.log_softmax(indexer_scores, dim=-1)
    else:
        k_select = min(topk, indexer_scores.size(-1))
        topk_idx = indexer_scores.topk(k_select, dim=-1).indices
        p = p.gather(dim=-1, index=topk_idx)
        p = p / p.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        I_s = indexer_scores.gather(dim=-1, index=topk_idx)
        log_q = F.log_softmax(I_s, dim=-1)

    # per-token KL: KL(p||q) summed over the block axis, kept per (B, S)
    kl_tok = F.kl_div(log_q, p, reduction='none').sum(dim=-1)        # (B, S)

    if mask is None:
        return kl_tok.mean()                                        # mean over B*S
    mask = mask.to(kl_tok.dtype)
    return (kl_tok * mask).sum() / mask.sum().clamp_min(1.0)         # mean over VALID tokens only


In [12]:
def freeze_for_dsa_warmup(model):
    """Freeze every parameter EXCEPT those whose name contains 'indexer'.
    Per paper §2.1 warmup stage: only the lightning indexer trains."""
    for name, param in model.named_parameters():
        param.requires_grad = "indexer" in name
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total     = sum(p.numel() for p in model.parameters())
    print(f"warmup freeze active: {n_trainable:,} / {n_total:,} trainable "
          f"({100*n_trainable/n_total:.2f}%)")

In [13]:
torch.manual_seed(0)
B, n = 2, 64                       # n=64 is divisible by both compress_rate(4) and hca_compress_rate(32)
H = torch.randn(B, n, config.hidden_size)

csa = TokenCompressor(config, compression_rate=4, overlapping=True)   # r=4  -> nb=16
hca = TokenCompressor(config, compression_rate=32, overlapping=False)  # r=32 -> nb=2

oc, oh = csa(H), hca(H)
assert oc.shape == (B, n // config.compress_rate,     config.compress_head_dim), oc.shape
assert oh.shape == (B, n // config.hca_compress_rate, config.compress_head_dim), oh.shape
print("CSA:", tuple(oc.shape), " HCA:", tuple(oh.shape))   # expect (2,16,64) (2,2,64)
assert torch.isfinite(oc).all() and torch.isfinite(oh).all(), "NaN/inf — check the entry-0 -inf padding"
print("shapes OK, finite OK")

# 2.4 equivalence check (r=2, tiny): hand-verify the overlap blends a-block i + b-block (i-1)
tc = TokenCompressor(config, compression_rate=2, overlapping=True)
h  = torch.randn(1, 6, config.hidden_size)
print("overlap out (1, 3, c):", tuple(tc(h).shape))

CSA: (2, 16, 64)  HCA: (2, 2, 64)
shapes OK, finite OK
overlap out (1, 3, c): (1, 3, 64)


In [14]:
# def repeat_kv(hidden_states, num_key_value_groups):
#     B, S, n_h, d_h = hidden_states.shape # at this moment, the k/v has been linearly projected in consideration of num_key_value_heads
#     if num_key_value_groups == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, :, None, :].expand(B, S, n_h, num_key_value_groups, d_h)
#     return hidden_states.reshape(B, S, n_h * num_key_value_groups, d_h)

# GPT architecture
class MLA(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.heads_dim = self.hidden_size // self.num_attention_heads # simpily using default ones instead of customized heads_dim
        self.num_key_value_heads = config.num_key_value_heads
        self.num_key_value_groups = self.num_attention_heads // self.num_key_value_heads
        self.dropout_prob = config.dropout
        self.flash_attn = self.config.flash_attn
        self.k_cache, self.v_cache = None, None
        self.is_causal = True
        self.dropout = nn.Dropout(self.dropout_prob) # simpily using the same value instead of distinguishing attention_dropout and residual_dropout

        self.q_lora_rank = config.q_lora_rank  # i.e. d_c'
        self.kv_lora_rank = config.kv_lora_rank # i.e. d_c
        self.qk_nope_head_dim = config.qk_nope_head_dim  # i.e. d_h
        self.qk_rope_head_dim = config.qk_rope_head_dim
        self.qk_head_dim = config.qk_nope_head_dim + config.qk_rope_head_dim # d_h + d^R_h
        self.v_head_dim = config.v_head_dim # d_h

        # down and up projection for mla
        self.wkv_a = nn.Linear(self.hidden_size, self.kv_lora_rank + self.qk_rope_head_dim) # down prj for hidden size, d->d_c+d^R_h; merge W^DKV and W^KR in one linear projection for easier computation
        self.kv_norm = RMSNorm(self.kv_lora_rank)
        self.wkv_b = nn.Linear(self.kv_lora_rank, self.num_attention_heads * (self.qk_nope_head_dim + self.v_head_dim))  # up prj for hidden size, d_c->n_h*(d_h+d_h); merge W^UK and W^UV in one linear projection for easier computation

        if self.q_lora_rank == 0:
            self.wq = nn.Linear(self.hidden_size, self.num_attention_heads * self.qk_head_dim)
        else:
            self.wq_a = nn.Linear(self.hidden_size, self.q_lora_rank) # down prj for hidden size, d->d_c'
            self.q_norm = RMSNorm(self.q_lora_rank)
            self.wq_b = nn.Linear(self.q_lora_rank, self.num_attention_heads * self.qk_head_dim) # up prj for hidden size, d_c'->n_h*(d_h+d^R_h)

        self.wo = nn.Linear(self.num_attention_heads * self.v_head_dim, self.hidden_size) # n_h*d_h->d
        self.rotary_emb = RotaryEmbedding(self.qk_rope_head_dim)

        if attn_impl == 'naive':
            self.register_buffer('k_cache', torch.zeros(config.max_batch_size, config.max_seq_len, self.num_attention_heads, self.qk_head_dim), persistent=False)
            self.register_buffer('v_cache', torch.zeros(config.max_batch_size, config.max_seq_len, self.num_attention_heads, self.v_head_dim), persistent=False)

        else:
            self.register_buffer('kv_cache', torch.zeros(config.max_batch_size, config.max_seq_len, self.kv_lora_rank), persistent=False)
            self.register_buffer('pe_cache', torch.zeros(config.max_batch_size, config.max_seq_len, self.qk_rope_head_dim), persistent=False)


    def forward(self, hidden_states, mask=None, use_kv_cache=False, start_pos=0):
        # `start_pos` is the absolute position of the FIRST query token in this batch.
        # Prefill: start_pos=0, S = prompt_len. (S == end_pos)
        # Decode step: start_pos = past_len, S = 1 (just the newly generated token). (end_pos = start_pos + 1)
        # Training: use_kv_cache=False, start_pos=0 — math identical to before this refactor.
        
        # NOTE: on q/k length -  q's length is always S (current batch). k's length is S (no cache)
        # or end_pos (with cache, reads full prefix). The einsum letters `s` (for q) and `t` (for k)
        # below encode exactly this asymmetry.
        B, S, d = hidden_states.shape
        end_pos = start_pos + S

        kv = self.wkv_a(hidden_states) # (B, S, d_c + d^R_h)
        kv_nope, k_pe = torch.split(kv, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1)
        if self.q_lora_rank == 0:
            q = self.wq(hidden_states)
        else:
            q = self.wq_a(hidden_states) # (B, S, d_c')
            q = self.q_norm(q) # (B, S, d_c')
            q = self.wq_b(q) # (B, S, n_h*(d_h+d^R_h))
        q = q.view(B, S, self.num_attention_heads, self.qk_head_dim) # (B, S, n_h, d_h+d^R_h)
        q_nope, q_pe = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)

        k_pe = k_pe.unsqueeze(2) # k_pe shape:(B, S, 1, d^R_h)
        q_pe, k_pe = self.rotary_emb(q_pe, k_pe, start_pos=start_pos)
        if attn_impl == 'naive':
            q = torch.cat([q_nope, q_pe], dim=-1) # (B, S, n_h, d_h+d^R_h)

            kv_nope = self.kv_norm(kv_nope)
            kv_nope = self.wkv_b(kv_nope) # (B, S, n_h*(d_h+d_h))
            kv_nope = kv_nope.view(B, S, self.num_attention_heads, self.qk_nope_head_dim + self.v_head_dim)
            k_nope, v_new = torch.split(kv_nope, [self.qk_nope_head_dim, self.v_head_dim], dim=-1)

            k_new = torch.cat([k_nope, k_pe.expand(-1,-1,self.num_attention_heads,-1)], dim=-1) # (B, S, n_h, d_h+d^R_h)

            if use_kv_cache:
                self.k_cache[:B, start_pos:end_pos, :, :] = k_new
                self.v_cache[:B, start_pos:end_pos, :, :] = v_new
                k = self.k_cache[:B, :end_pos]  # (B, end_pos, n_h, qk_head_dim) -- full prefix incl. k_new
                v = self.v_cache[:B, :end_pos]  # (B, end_pos, n_h, v_head_dim)
            else:
                k, v = k_new, v_new              # (B, S, n_h, qk_head_dim) / (B, S, n_h, v_head_dim)

            scores = torch.einsum("bshd,bthd->bsht", q, k) / math.sqrt(self.qk_head_dim)
        else:
            # consider weights absortion to avoid calculating k distinctly, i.e. via changing multiply order i.e. A*(B*C) -> (A*B) * C to reduce computation cost
            wkv_b = self.wkv_b.weight # if self.wkv_b.scale is None else weight_dequant(self.wkv_b.weight, self.wkv_b.scale, block_size) , (d_h*n_h, d_c)
            wkv_b = wkv_b.view(self.num_attention_heads, -1, self.kv_lora_rank) # (n_h, d_h, d_c)
            # q_{nope} = q_{nope} \times W^{UK}
            q_nope = torch.einsum("bshd,hdc->bshc", q_nope, wkv_b[:, :self.qk_nope_head_dim])
            kv_nope_new = self.kv_norm(kv_nope)
            k_pe_2d_new = k_pe.squeeze(2)

            if use_kv_cache:
                self.kv_cache[:B, start_pos:end_pos] = kv_nope_new
                self.pe_cache[:B, start_pos:end_pos] = k_pe_2d_new
                kv_nope = self.kv_cache[:B, :end_pos]  # (B, end_pos, kv_lora_rank)
                k_pe_2d = self.pe_cache[:B, :end_pos]  # (B, end_pos, qk_rope_head_dim)
            else:
                kv_nope = kv_nope_new                  # (B, S, kv_lora_rank)
                k_pe_2d = k_pe_2d_new                  # (B, S, qk_rope_head_dim)

            scores = (torch.einsum("bshc,btc->bsht", q_nope, kv_nope) +
                      torch.einsum("bshr,btr->bsht", q_pe, k_pe_2d)) / math.sqrt(self.qk_head_dim)
        if mask is not None:
            scores += mask.unsqueeze(1)
        scores = scores.softmax(dim=-1)

        if attn_impl == 'naive':
            x = torch.einsum("bsht,bthd->bshd", scores, v)  # (B, S, n_h, d_h)
        else:
            # `kv_nope` is the cache slice during decode and the live latent during training.
            # Either way it carries the right K-side rows for the weighted sum.
            x = torch.einsum("bsht,btc->bshc", scores, kv_nope)
            x = torch.einsum("bshc,hdc->bshd", x, wkv_b[:, -self.v_head_dim:])
        x = self.wo(x.flatten(2))
        return x

In [15]:
def precompute_freqs_cis(dim, seqlen, theta=10000.0):
    """Complex rotary table of shape (seqlen, dim//2). Simplified (no YaRN)."""
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(seqlen).float()
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)


class Attention(nn.Module):
    """Unified compressed attention (DeepSeek-V4 §2.3), switched by `compression_rate`:
        rate == hca_compress_rate -> HCA: heavy compression, dense over all visible blocks, no indexer.
        rate == compress_rate     -> CSA: light overlapping compression + lightning-indexer top-k over blocks.
    Core (shared): shared-KV MQA over [sliding-window ; compressed blocks] + grouped output projection,
    with Q/KV RMSNorm, partial RoPE (+ the -i output trick) and an attention sink.
    """
    def __init__(self, config: Config, compression_rate: int):
        super().__init__()
        self.config = config
        self.d = config.hidden_size
        self.nh = config.compress_n_heads
        self.c  = config.compress_head_dim
        self.rd = config.compress_rope_dim
        self.m  = compression_rate
        self.is_csa = (compression_rate == config.compress_rate)   # else HCA
        self.block_topk = config.index_block_topk
        self.n_win = config.sliding_window
        self.use_sink = config.use_attention_sink
        self.g  = config.o_groups
        self.dg = config.o_lora_rank

        self.token_compressor = TokenCompressor(config, compression_rate, overlapping=self.is_csa)
        self.w_kv_win = nn.Linear(self.d, self.c, bias=False)    # sliding window entry (1 KV head)
        # low-rank query (optional). With q_lora_rank==0, project hidden -> nh*c directly.
        if config.q_lora_rank == 0:
            self.wq_a, self.q_norm = None, None
            self.wq_b = nn.Linear(self.d, self.nh * self.c, bias=False)
        else:
            self.wq_a = nn.Linear(self.d, config.q_lora_rank, bias=False)
            self.q_norm = RMSNorm(config.q_lora_rank)                        # norm on the q-latent
            self.wq_b = nn.Linear(config.q_lora_rank, self.nh * self.c, bias=False)
        self.q_norm_b = RMSNorm(self.c)
        self.win_kv_norm  = RMSNorm(self.c)
        self.comp_kv_norm = RMSNorm(self.c)
        self.wo_a = nn.Linear((self.nh * self.c) // self.g, self.dg, bias=False)   # per-group
        self.wo_b = nn.Linear(self.g * self.dg, self.d, bias=False)
        if self.use_sink:
            self.sink = nn.Parameter(torch.zeros(self.nh))      # learnable per-head sink logit z'_h
        self.register_buffer("freqs_cis", precompute_freqs_cis(self.rd, config.max_seq_len), persistent=False)

        # lightning indexer: CSA + an active DSA stage only. Stashes feed the indexer KL.
        self.indexer = Indexer(config, self.m) if (self.is_csa and config.dsa_stage != 'off') else None
        self.last_attn_probs = None
        self.last_indexer_scores = None

        # kv cache
        self.register_buffer("win_cache", torch.zeros(self.config.max_batch_size, self.config.max_seq_len, self.c), persistent=False)

    @staticmethod
    def _apply_rotary_emb(x, freqs_cis, inverse=False):
        """Rotate the LAST dim of `x` (caller slices the rope dims, e.g. x[..., -rd:]).
        inverse=True rotates by -pos (conjugate) for the output `-i` trick.
        Non-in-place so it is autograd-safe (the official does an in-place copy_ because
        it runs under inference_mode)."""
        dtype = x.dtype
        xc = torch.view_as_complex(x.float().unflatten(-1, (-1, 2)))   # (..., rd//2)
        if inverse:
            freqs_cis = freqs_cis.conj()
        if xc.ndim == 3:                      # (B, T, rd//2)       compressed / window kv
            freqs_cis = freqs_cis.view(1, xc.size(1), xc.size(-1))
        else:                                 # (B, S, nh, rd//2)   queries / output
            freqs_cis = freqs_cis.view(1, xc.size(1), 1, xc.size(-1))
        return torch.view_as_real(xc * freqs_cis).flatten(-2).to(dtype)

    @staticmethod
    def _get_window_topk_idxs(window_size, bsz, seqlen, start_pos=0):
        """Per-query sliding-window key indices (bsz, seqlen, <=window_size); -1 = padding (not selected).
        Query i attends tokens [max(0, i-window+1) .. i]."""
        if start_pos == 0:
            # prefill phase
            base = torch.arange(seqlen).unsqueeze(1)  # (S, 1)
            matrix = (base - window_size + 1).clamp(0) + torch.arange(min(seqlen, window_size))
            matrix = torch.where(matrix > base, -1, matrix)  # future -> -1, (S, <=window)
            return matrix.unsqueeze(0).expand(bsz, -1, -1)   # (B, S, <=window)
        else:
            # decode phase, S = 1
            end_pos = start_pos + seqlen
            cols = torch.arange(max(0, end_pos-window_size), end_pos)
            return cols.view(1, 1, -1).expand(bsz, 1, -1)
    

    @staticmethod
    def _get_compress_topk_idxs(compress_rate, bsz, seqlen, start_pos, offset):
        """Per-query visible compressed-block indices (+offset) (bsz, seqlen, nb); -1 = invisible (causal).
        Block s visible to query i iff  s < (i+1)//ratio  (block fully completed by i).
        `offset` shifts these into the compressed region of the unified KV list."""
        if start_pos == 0:
            matrix = torch.arange(seqlen // compress_rate).repeat(seqlen, 1)  # (S, nb)
            mask = matrix >= torch.arange(1, seqlen + 1).unsqueeze(1) // compress_rate  # (S, 1) broadcast
            matrix = torch.where(mask, -1, matrix + offset)
            return matrix.unsqueeze(0).expand(bsz, -1, -1)   # (B, S, nb)
        else:
            nb = (start_pos + seqlen) // compress_rate
            cols = torch.arange(nb) + offset
            return cols.view(1, 1, -1).expand(bsz, 1, -1)



    def forward(self, hidden_states, mask=None, use_kv_cache=False, start_pos=0):
        # `mask` (token causal) is accepted for signature parity but unused here: block/window
        # causality is carried by the -1 entries of comp_idx / win_idx instead.

        # NOTE : about kv cache in decode phase - csa mode cache {window, compressed, state, indexer-key};
        # hca mode cache {window, compressed, state},mla mode cache {kv, pe} as what implemented before,
        # where state tracks the unready tail tokens for compression
        B, S, _ = hidden_states.shape
        end_pos = start_pos + S
        dev = hidden_states.device
        ff = self.freqs_cis.to(dev)
        fq = ff[start_pos:end_pos]                      # query / window token positions

        # queries: nh heads of dim c; per-head RMSNorm; partial RoPE on last rd
        qr = self.wq_a(hidden_states) if self.wq_a is not None else hidden_states
        if self.q_norm is not None:
            qr = self.q_norm(qr)
        q = self.wq_b(qr).view(B, S, self.nh, self.c)
        q = self.q_norm_b(q)
        q = torch.cat([q[..., :-self.rd], self._apply_rotary_emb(q[..., -self.rd:], fq)], dim=-1)

        # window KV (one uncompressed entry per token, single shared head)
        win_kv_new = self.win_kv_norm(self.w_kv_win(hidden_states))
        win_kv_new = torch.cat([win_kv_new[..., :-self.rd], self._apply_rotary_emb(win_kv_new[..., -self.rd:], fq)], dim=-1)

        if use_kv_cache:
            self.win_cache[:B, start_pos:end_pos] = win_kv_new
            win_kv = self.win_cache[:B, :end_pos]
        else:
            win_kv = win_kv_new
        # compressed KV
        comp_kv = self.comp_kv_norm(self.token_compressor(hidden_states, use_kv_cache, start_pos))
        nb = comp_kv.shape[1]
        if nb > 0:
            fb = ff[torch.arange(nb, device=dev) * self.m]
            comp_kv = torch.cat([comp_kv[..., :-self.rd], self._apply_rotary_emb(comp_kv[..., -self.rd:], fb)], dim=-1)

        # per-query visible compressed-block indices into `kv`
        comp_idx = self._get_compress_topk_idxs(self.m, B, S, start_pos, offset=win_kv.shape[1]).to(dev)   # (B, S, nb), where offset is because concat win_kv later, so need to offset by win_kv.col which is up to end_pos

        # CSA: lightning-indexer top-k over blocks (sparse stage) + stash scores for the KL
        if self.indexer is not None:
            # detach indexer inputs in the sparse stage so KL grad can't flow into the main model.
            # In dense warmup the main params are frozen, so no detach is needed.
            if self.config.dsa_stage == 'sparse':
                h_idx  = hidden_states.detach()
                qr_idx = qr.detach() if qr is not None else None
            else:
                h_idx, qr_idx = hidden_states, qr
            I = self.indexer(h_idx, qr=qr_idx, use_kv_cache=use_kv_cache, start_pos=start_pos)   # (B, S, nb)

            if self.config.dsa_stage == 'sparse' and nb > 0:
                # rank only visible blocks, keep the top-k, drop the rest (index -> -1)
                I_vis = I.masked_fill(comp_idx < 0, float('-inf'))
                topk_blk = I_vis.topk(min(self.block_topk, nb), dim=-1).indices              # (B, S, k)
                keep = torch.zeros_like(comp_idx, dtype=torch.bool).scatter_(-1, topk_blk, True)
                comp_idx = torch.where(keep & (comp_idx >= 0), comp_idx, torch.full_like(comp_idx, -1))
            self.last_indexer_scores = I

        # unified KV list = [window (S) ; compressed (nb)]  (window first, as in official)
        kv = torch.cat([win_kv, comp_kv], dim=1) # (B, S+nb, c)

        # per-query selection indices into `kv` (window region offset 0, compressed offset S)
        win_idx  = self._get_window_topk_idxs(self.n_win, B, S, start_pos).to(dev)   # (B, S, <=n_win)
        topk_idx = torch.cat([win_idx, comp_idx], dim=-1)                            # (B, S, K)

        # gather selected keys/values (MQA: one KV head shared across query heads)
        valid = topk_idx >= 0
        idx = topk_idx.clamp(min=0)
        kv_g = kv[torch.arange(B, device=dev).view(B, 1, 1), idx]                    # (B, S, K, c)
        scores = torch.einsum("bshc,bskc->bshk", q, kv_g) / math.sqrt(self.c)        # (B, S, nh, K)
        scores = scores.masked_fill(~valid.unsqueeze(2), float('-inf'))             # padding -> no attention

        # attention sink: append a per-head sink column (value 0), softmax, drop it
        if self.use_sink:
            scores = torch.cat([scores, self.sink.view(1, 1, self.nh, 1).expand(B, S, self.nh, 1)], dim=-1)
        probs = scores.softmax(dim=-1)
        if self.use_sink:
            probs = probs[..., :topk_idx.size(-1)]                                   # drop the sink column

        # stash the compressed-BLOCK probs (post-softmax) for the indexer KL - aligns with I (B,S,nb).
        # Block region is the LAST nb columns of the unified key list (window first, blocks last).
        if self.indexer is not None and nb > 0:
            self.last_attn_probs = probs[..., -nb:].detach()                         # (B, S, nh, nb)

        o = torch.einsum("bshk,bskc->bshc", probs, kv_g)                             # (B, S, nh, c)
        # -i output RoPE: the gathered value carried +pos (it IS the key); de-rotate by -t
        o = torch.cat([o[..., :-self.rd], self._apply_rotary_emb(o[..., -self.rd:], fq, inverse=True)], dim=-1)

        # grouped low-rank output projection
        o = o.reshape(B, S, self.g, (self.c * self.nh) // self.g)                    # (B,S,nh,c) -> (B,S,g,nh*c/g)
        o = self.wo_a(o).reshape(B, S, self.g * self.dg)                            # -> (B,S,g,dg) -> (B,S,g*dg)
        o = self.wo_b(o)
        return o                                                                    # (B, S, d)

In [16]:
# matrix: 
# [[1, 2, ...nb],
# [1, 2, ...nb],
# [1, 2, ...nb]
# ...,
# [1, 2, ...nb]] # (S, nb)

# mask: 
# [[0, -1, -1, -1, ... -1],
#  [0, 0, -1, -1, ... -1],
#  [0, 0, 0, -1, ... -1],
#  ...
#  [0, 0, 0, 0, ... -1],]

# matrix+offset because S -> S+nb, so offset=S

In [17]:
torch.manual_seed(0)
hca = Attention(config, compression_rate=config.hca_compress_rate)
for S in [64, 16, 100]:                      # 16 < m'(32) -> nb=0, sink must keep it finite
    out = hca(torch.randn(2, S, config.hidden_size))
    assert out.shape == (2, S, config.hidden_size) and torch.isfinite(out).all()
print("HCA OK")

HCA OK


In [18]:
# CSA smoke test - all DSA stages + black-box causality (future tokens must not affect earlier outputs)
torch.manual_seed(0)
for stage in ['off', 'dense_warmup', 'sparse']:
    cfg = Config(dsa_stage=stage)
    a = Attention(cfg, cfg.compress_rate).eval()      # CSA layer (rate == compress_rate)
    B, S = 2, 64
    h = torch.randn(B, S, cfg.hidden_size)
    out = a(h)
    assert out.shape == (B, S, cfg.hidden_size) and torch.isfinite(out).all(), stage
    if stage != 'off':
        nb = S // cfg.compress_rate
        assert a.last_indexer_scores.shape == (B, S, nb), a.last_indexer_scores.shape
        assert a.last_attn_probs.shape == (B, S, cfg.compress_n_heads, nb), a.last_attn_probs.shape
    # causality: perturb the second half; outputs before the cut must be unchanged
    P = 32
    with torch.no_grad():
        h2 = h.clone(); h2[:, P:] += 5.0
        assert torch.allclose(a(h)[:, :P], a(h2)[:, :P], atol=1e-5), f"causality broken @ {stage}"
    print(f"CSA[{stage}] OK out={tuple(out.shape)}")
print("CSA OK (shapes + stashes + causality)")

CSA[off] OK out=(2, 64, 512)
CSA[dense_warmup] OK out=(2, 64, 512)
CSA[sparse] OK out=(2, 64, 512)
CSA OK (shapes + stashes + causality)


In [19]:
# config = Config(dsa_stage='sparse')
# mla = MLA(config)
# hidden_states = torch.randn(config.max_batch_size, 512, config.hidden_size)
# u = mla(hidden_states, mask=None, use_kv_cache=False, start_pos=0)
# assert u.shape == (config.max_batch_size, 512, config.hidden_size), 'attention output shape is wrong'

In [20]:
class FeedForward(nn.Module):
    # Note: why not using nn.Sequential() to implement SwiGLU? - cuz it's not linear pipeline but including parallel structure and a multiplicative operation
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size
        self.dropout = config.dropout
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=config.mlp_bias)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=config.mlp_bias)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=config.mlp_bias)

    def forward(self, hidden_states):
        down_proj = self.down_proj(F.silu(self.gate_proj(hidden_states)) * self.up_proj(hidden_states))
        return down_proj


In [21]:
class DecoderLayer(nn.Module):
    def __init__(self, config: Config, attn_type):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.attn_type = attn_type
        if self.attn_type == 'csa':
            self.attention = Attention(config, config.compress_rate)
        elif self.attn_type == 'hca':
            self.attention = Attention(config, config.hca_compress_rate)
        else:
            self.attention = MLA(config)
        self.ffn = FeedForward(config)
        self.input_layernorm = RMSNorm(self.hidden_size)
        self.post_attention_layernorm = RMSNorm(self.hidden_size)

    def forward(self, hidden_states, mask=None, use_kv_cache=False, start_pos=0):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.attention(hidden_states, mask=mask, use_kv_cache=use_kv_cache, start_pos=start_pos)
        hidden_states = residual + hidden_states
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.ffn(hidden_states)
        hidden_states = residual + hidden_states
        return hidden_states

In [22]:
class LLM(PreTrainedModel):
    config_class = Config  # for later on: AutoModelForCausalLM.register(Config, LLM)
    def __init__(self, config):
        super().__init__(config)
        self.vocab_size = self.config.vocab_size
        self.n_layers = self.config.n_layers
        self.dropout = nn.Dropout(self.config.dropout)
        self.token_embeddings = nn.Embedding(self.vocab_size, self.config.hidden_size)
        self.layers = torch.nn.ModuleList()
        for _, attn_type in zip(range(self.n_layers), config.attn_schedule):
            self.layers.append(DecoderLayer(config, attn_type))
        self.layernorm = RMSNorm(self.config.hidden_size)
        self.output = nn.Linear(self.config.hidden_size, self.vocab_size, bias=False) # each token generated's shape is (hidden_size, vocab_size)
        self.apply(self._init_weights)
        self.loss = None

        # TODO: Have no idea why it looks like this, looks so hacky - explained by GPT:
        # the loop over self.named_parameters() looks for tensor names ending with w3.weight (the MLP’s down-projection in a SwiGLU block)
        # or wo.weight (the attention output projection) and rescales them with a smaller std, 0.02 / sqrt(2 * n_layers), to match the RMSNorm-residual scaling used in LLaMA-style models.
        for pn, p in self.named_parameters():
            if pn.endswith('w3.weight') or pn.endswith('wo.weight') or pn.endswith('down_proj.weight') or pn.endswith('wo_b.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * self.config.n_layers))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


    def forward(self, input_ids, labels=None, use_kv_cache=False, start_pos=0):
        B, S = input_ids.shape
        end_pos = start_pos + S
        kl_mask = (input_ids != 0)
        hidden_states = self.token_embeddings(input_ids)
        hidden_states = self.dropout(hidden_states)
        # Causal mask of shape (S_q, T_k). Query i is at absolute position (start_pos+i)
        # and may attend to key positions [0, start_pos+i]. T_k is end_pos when caching
        # (keys are the full prefix held in cache), else S (training: keys == queries).
        # Mask broadcasts to scores (B, S_q, n_h, T_k) via mask.unsqueeze(1) inside MLA.
        if S == 1 and start_pos > 0:
            # Decode step: the single new query at position start_pos can attend to
            # every cached key 0..start_pos — no positions to mask out.
            mask = None
        else:
            T_k = end_pos if use_kv_cache else S
            row = torch.arange(S, device=input_ids.device).unsqueeze(1)        # (S, 1)
            col = torch.arange(T_k, device=input_ids.device).unsqueeze(0)      # (1, T_k)
            mask = torch.where(col <= start_pos + row, 0.0, float('-inf'))
        for layer in self.layers:
            hidden_states = layer(hidden_states, mask=mask, use_kv_cache=use_kv_cache, start_pos=start_pos)
        hidden_states = self.layernorm(hidden_states)

        if labels is not None:
            logits = self.output(hidden_states)
            main_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=0)
            if self.config.dsa_stage != 'off':
                kl_loss_total = 0.0
                valid_layers = 0
                for layer in self.layers:
                    if layer.attn_type == 'csa':
                        attn = layer.attention
                        kl_loss = compute_indexer_kl(attn.last_attn_probs, attn.last_indexer_scores, self.config.dsa_stage, self.config.index_block_topk, mask=kl_mask)
                        kl_loss_total += kl_loss
                        valid_layers += 1

                if valid_layers > 0:
                    self.loss = main_loss + self.config.indexer_kl_weight * (kl_loss_total / valid_layers)
                else:
                    self.loss = main_loss
            else:
                self.loss = main_loss
        else:
            # for inference
            logits = self.output(hidden_states[:, [-1], :])
            self.loss = None

        return CausalLMOutputWithPast(self.loss, logits) # meaning can call LLM().loss, LLM.logits directly

    @torch.inference_mode
    def generate(self, inputs, eos, max_new_tokens, temperature=0.7, top_k=None, stream=True, repetition_penalty=1.,
                 use_kv_cache=True):

        input_ids = inputs['input_ids']
        s = input_ids.shape[1]
        # start_pos = how many tokens are already represented in the KV cache.
        # First iteration: 0  -> forward the entire prompt (the "prefill" pass).
        # Subsequent iterations: forward only the single new token at position start_pos.
        # Without cache: start_pos stays 0 and we forward the full growing sequence each step
        # (quadratic-cost behavior, kept as a comparison baseline).
        start_pos = 0
        while input_ids.shape[1] < max_new_tokens - 1:
            tokens_in = input_ids[:, start_pos:] if use_kv_cache else input_ids
            inference_res = self.forward(tokens_in, labels=None,
                                         use_kv_cache=use_kv_cache, start_pos=start_pos)
            logits = inference_res.logits
            logits = logits[:, -1, :]

            # apply penaly for repetitive tokens
            for token in set(input_ids.tolist()[0]):
                logits[:, token] /= repetition_penalty

            if temperature == 0.0:
                _, idx_next = torch.topk(logits, k=1, dim=-1)
            else:
                logits = logits / temperature
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float('Inf')

                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1, generator=None)

            if idx_next == eos:
                break

            # IMPORTANT: advance start_pos BEFORE we cat the new token onto input_ids,
            # so the next iteration's `input_ids[:, start_pos:]` is just the new token.
            if use_kv_cache:
                start_pos = input_ids.shape[1]

            input_ids = torch.cat((input_ids, idx_next), dim=1)
            if stream:
                yield input_ids[:, s:]

        if not stream:
            yield input_ids[:, s:] 

In [23]:
# Cache-parity test (roadmap 6.6 / Phase 7): incremental decode must match one-shot prefill.
# Validates the whole KV-cache path - window cache + compressed/state cache + indexer-key cache
# (+ MLA's kv/pe cache) - by decoding token-by-token and comparing to an independent full prefill.
@torch.no_grad()
def cache_parity(schedule, stage, S=40, P=20):
    torch.manual_seed(0)
    cfg = Config(dsa_stage=stage, n_layers=len(schedule), attn_schedule=schedule,
                 max_seq_len=64, max_batch_size=1, compress_rate=4, hca_compress_rate=8, sliding_window=8)
    m = LLM(cfg).eval()
    ids = torch.randint(1, cfg.vocab_size, (1, S))
    m(ids[:, :P], use_kv_cache=True, start_pos=0)                                          # prefill (seeds caches)
    inc = [m(ids[:, t:t+1], use_kv_cache=True, start_pos=t).logits[:, -1] for t in range(P, S)]  # decode
    worst = max((m(ids[:, :t+1], use_kv_cache=False).logits[:, -1] - inc[k]).abs().max().item()
                for k, t in enumerate(range(P, S)))                                        # vs one-shot prefill
    print(f"  [{'/'.join(schedule):17s} {stage:12s}] max|diff| = {worst:.2e}")
    return worst

# the cache machinery is bit-exact for every path that does NOT do discrete top-k
for sched, stage in [(['hca', 'hca'], 'off'), (['csa', 'csa'], 'off'),
                     (['hca', 'hca', 'csa', 'hca'], 'off'), (['mla', 'hca', 'csa', 'mla'], 'off')]:
    assert cache_parity(sched, stage) < 1e-4, "cache parity FAILED"
print("cache parity OK - window + compressed/state + indexer-key + MLA caches are bit-exact")

# CSA-sparse: exact EXCEPT where the indexer's top-k hits a score tie. ReLU saturates several
# block scores to exactly 0.0 (at random init), so the top-k boundary is ambiguous and torch.topk
# breaks the tie differently for prefill (n blocks) vs decode (n+1 in-array). This is a discrete-
# selection artifact, NOT a cache bug - DeepSeek uses batch-invariant deterministic kernels (paper
# 3.3) to keep prefill/decode selection consistent.
d = cache_parity(['csa', 'csa'], 'sparse')
print(f"  ^ CSA-sparse residual ({d:.1e}) is a top-k tie-break, not a cache error (dense paths above are exact)")

  [hca/hca           off         ] max|diff| = 3.16e-06
  [csa/csa           off         ] max|diff| = 3.17e-06
  [hca/hca/csa/hca   off         ] max|diff| = 3.96e-06
  [mla/hca/csa/mla   off         ] max|diff| = 4.02e-06
cache parity OK - window + compressed/state + indexer-key + MLA caches are bit-exact
  [csa/csa           sparse      ] max|diff| = 7.99e-02
  ^ CSA-sparse residual (8.0e-02) is a top-k tie-break, not a cache error (dense paths above are exact)


In [24]:
tokenizer = AutoTokenizer.from_pretrained("../../sft/tokenizer")
tokenizer.bos_token = '<|im_start|>' # based on original data
tokenizer.eos_token = '<|im_end|>'
print("vocab size:", len(tokenizer), " bos:", tokenizer.bos_token_id, " eos:", tokenizer.eos_token_id)
tokenizer.add_special_tokens({'additional_special_tokens': ['<|im_start|>', '<|im_end|>']})
tokenizer.convert_tokens_to_ids(['<|im_start|>', '<|im_end|>'])

vocab size: 6400  bos: 1  eos: 1


[6400, 6401]

In [25]:
class LLMDataset(IterableDataset):
    """Pretraining dataset for the custom LLM (small tokenizer, minimind-style).

    Pads with token id 0 in BOTH input_ids and labels. The model's CE loss uses
    ignore_index=0, so padded positions don\'t contribute to loss.

    Pre-shifts: X = ids[:-1], Y = ids[1:]. Our custom LLM.forward does NOT auto-shift
    internally (unlike HF Qwen2ForCausalLM).
    """
    def __init__(self, data_path, tokenizer, max_seq_len):
        super().__init__()
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __iter__(self):
        return self.data_process()

    def data_process(self):
        with open(self.data_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = json.loads(line)
                text = line['text']
                input_ids = self.tokenizer.encode(text)
                text_len = len(input_ids)
                if text_len > self.max_seq_len:
                    input_ids = input_ids[:self.max_seq_len]
                else:
                    input_ids = input_ids + [0] * (self.max_seq_len - text_len)
                input_ids = np.array(input_ids)
                X = np.array(input_ids[:-1]).astype(np.int64)
                Y = np.array(input_ids[1:]).astype(np.int64)
                yield {
                    'input_ids': torch.from_numpy(X),
                    'labels':    torch.from_numpy(Y),
                }

In [26]:
dataset = LLMDataset("../../sft/dataset/pretrain_hq.jsonl", tokenizer, max_seq_len=512)

In [27]:
# Instantiate the custom MLA-based LLM from scratch. Uses the Config defined above
# (vocab=6400, hidden=512, 8 layers, MLA with kv_lora_rank=128). All params trainable.
config = Config(dsa_stage='off')
model = LLM(config)
print(f"trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

trainable params: 35,477,056


In [28]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(model)

35477056

In [29]:
class DSATrainer(Trainer):
    """HF Trainer subclass with two AdamW param groups: main model + indexer.
    The indexer LR is read from `args.indexer_lr` (custom attribute); the main LR
    uses the standard `args.learning_rate`. Per paper §2.1:
        warmup: indexer 1e-3, main frozen (LR irrelevant)
        sparse: main 7.3e-6, indexer ~1e-4 (paper doesn't specify)
    """
    def create_optimizer(self):
        if self.optimizer is None:
            indexer_params = [p for n, p in self.model.named_parameters()
                              if 'indexer' in n and p.requires_grad]
            main_params    = [p for n, p in self.model.named_parameters()
                              if 'indexer' not in n and p.requires_grad]

            main_lr    = self.args.learning_rate
            indexer_lr = getattr(self.args, 'indexer_lr', main_lr)

            self.optimizer = torch.optim.AdamW([
                {'params': main_params,    'lr': main_lr,    'weight_decay': 0.0},
                {'params': indexer_params, 'lr': indexer_lr, 'weight_decay': 0.0},
            ])
            print(f"DSATrainer optimizer: main_lr={main_lr}, indexer_lr={indexer_lr}, "
                  f"#main={sum(p.numel() for p in main_params):,}, "
                  f"#indexer={sum(p.numel() for p in indexer_params):,}")
        return self.optimizer


In [30]:
# ============================================================
# Stage 0: Main model pre-training w new architecture
# ============================================================

DSA_STAGE = 'off'

config = Config(dsa_stage=DSA_STAGE, vocab_size=len(tokenizer))
model = LLM(config)

print(f"trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

main_lr = 5e-4

args = TrainingArguments(
    output_dir='./v4/start',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    max_steps=500,                  # paper: 1000; smaller here to iterate quickly
    logging_steps=50,
    save_steps=500,
    learning_rate=main_lr,
    report_to='none',
)

trainer = DSATrainer(
    model=model, args=args, train_dataset=dataset,
    processing_class=tokenizer, data_collator=DefaultDataCollator(),
)
trainer.train(resume_from_checkpoint=False)

trainer.save_model('./model/v4_start')
print('Stage 1 complete. Starter checkpoint saved to ./model/v4_start')

trainable params: 35,479,104


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 6401, 'bos_token_id': 6400}.
/Users/yingyao/miniconda3/envs/transformer-practice/lib/python3.14/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


DSATrainer optimizer: main_lr=0.0005, indexer_lr=0.0005, #main=35,479,104, #indexer=0


Step,Training Loss
50,6.149500
100,5.239100
150,5.034800
200,4.755000
250,5.623000
300,4.744400
350,4.283900
400,4.695800
450,4.586700
500,4.153600


Stage 1 complete. Starter checkpoint saved to ./model/v4_start


In [ ]:
# ============================================================
# Stage 1: Dense warmup — Indexer pre-training
# ============================================================

from safetensors.torch import load_file

DSA_STAGE = 'dense_warmup'

config = Config(dsa_stage=DSA_STAGE, vocab_size=len(tokenizer))
model = LLM(config)

# Load pretrained weights into the new DSA-shaped model.
mla_state = load_file('./model/v4_start/model.safetensors')
missing, unexpected = model.load_state_dict(mla_state, strict=False)
assert all('indexer' in k for k in missing), \
    f"unexpected non-indexer missing keys: {[k for k in missing if 'indexer' not in k]}"
assert not unexpected, f"unexpected keys (should be 0): {unexpected}"
print(f"loaded pre-training weights — {len(missing)} indexer params kept at random init")

# Freeze main; only indexer is trainable
freeze_for_dsa_warmup(model)

indexer_lr = 1e-3      # paper §2.1
main_lr    = 0.0       # frozen anyway, value irrelevant

args = TrainingArguments(
    output_dir='./v4/warmup',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    max_steps=200,                  # paper: 1000; smaller here to iterate quickly
    logging_steps=20,
    save_steps=200,
    learning_rate=main_lr,
    report_to='none',
)
args.indexer_lr = indexer_lr

trainer = DSATrainer(
    model=model, args=args, train_dataset=dataset,
    processing_class=tokenizer, data_collator=DefaultDataCollator(),
)
trainer.train(resume_from_checkpoint=False)

trainer.save_model('./model/v4_warmup')
print('Stage 1 complete. Warmed-up checkpoint saved to ./model/v4_warmup')

In [ ]:
# ============================================================
# Stage 2: Sparse training —Indexer and main model continuous training
# ============================================================

from safetensors.torch import load_file

DSA_STAGE = 'sparse'

config = Config(dsa_stage=DSA_STAGE, vocab_size=len(tokenizer))
model = LLM(config)

# Load full warmup state (main + trained indexer)
warmup_state = load_file('./model/v4_warmup/model.safetensors')
missing, unexpected = model.load_state_dict(warmup_state, strict=True)
print(f"loaded warmup checkpoint: 0 missing, 0 unexpected — OK")

indexer_lr = 1e-4
main_lr    = 7.3e-6    # paper §2.1

args = TrainingArguments(
    output_dir='./v4/sparse',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    max_steps=500,                  # paper: 15000; smaller here to iterate quickly
    logging_steps=50,
    save_steps=500,
    learning_rate=main_lr,
    report_to='none',
)
args.indexer_lr = indexer_lr

trainer = DSATrainer(
    model=model, args=args, train_dataset=dataset,
    processing_class=tokenizer, data_collator=DefaultDataCollator(),
)
trainer.train(resume_from_checkpoint=False)

trainer.save_model('./model/v4_sparse')
print('Stage 2 complete. Final DSA model saved to ./model/v4_sparse')

In [ ]:
# Reload the final DSA-trained checkpoint via the HF Auto* registry.
# Change the path to load a different stage (e.g., './model/dsa_warmup').
AutoConfig.register("v4_replica", Config)
AutoModelForCausalLM.register(Config, LLM)
reload_model = AutoModelForCausalLM.from_pretrained('./model/v4_sparse')

# Build a short prompt to test generation.
input_ids = [tokenizer.bos_token_id] + tokenizer.encode("1+1等于几?")
input_data = {'input_ids': torch.tensor(input_ids).unsqueeze(0), "labels": None}
input_data

In [ ]:
# Use LLM.generate (our custom generator method) — yields the suffix tokens as it goes.
# With only 100 training steps the output will be gibberish; you mostly want to see that
# the cache + sparse-attention machinery doesn't crash.
for token in reload_model.generate(inputs=input_data, eos=tokenizer.eos_token_id,
                                   max_new_tokens=100, stream=False):
    print(tokenizer.decode(token[0]))

## Appendix

In [ ]:
# rmsnorm = RMSNorm(hidden_size=768)
# torch.manual_seed(1234)
# result = rmsnorm.forward(torch.randn(1, 1024, 768))
# # print(rmsnorm.weight)
# print(result)

In [ ]:
# import torch
# dim = 768
# max_seq_len=2048
# inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))  # 形状(dim/2)
# t = torch.arange(max_seq_len).float().unsqueeze(1)  # 形状(max_seq_len, 1)

# freqs = t @ inv_freq.unsqueeze(0)  #(max_seq_len, 1)*(1, dim/2) = (max_seq_len, dim/2)
# print(t)
# print(freqs.shape)

In [ ]:
# k = torch.randn((2, 3, 4, 5))
# q = torch.randn((2, 3, 4, 5))
# v = torch.randn((2, 3, 4, 5))
# mask = torch.randn((2, 3, 6, 6))
# mask = torch.triu(mask, diagonal=1)
# scores = torch.matmul(q, k.transpose(2, 3)) / math.sqrt(5) 

In [ ]:
# # encoded input looks like: 
# data_iter = iter(dataset)
# print(type(data_iter))
# sample = next(data_iter)
# input_ids = sample['input_ids']
# tokenizer.decode(input_ids)

In [ ]:
# input_ids = torch.randint(0, 10, (2, 512)) # min_int, max_int, (S, H)
# labels = torch.randint(0, 10, (2, 512))
# model(input_ids, labels).logits.shape

In [ ]:
hidden_states = torch.randn(2, 100, 512)
attn_impl: Literal["naive", "absorb"] = "absorb"
mla = MLA(config=config)

# print(mla(hidden_states))
assert mla(hidden_states).shape == hidden_states.shape, 'attention output shape is wrong'
# print(mla.kv_cache)
assert mla.kv_cache.shape == torch.Size([2, config.max_seq_len, config.kv_lora_rank]), 'kv_cache shape is wrong'

In [ ]:
##